# Data quality before classifier tuning

Read completed server artifacts only. No PCA, MI, bootstrap or classifier is fitted here.
Set `IIVV_QUALITY` to the copied quality directory. Whole-cohort projections are descriptive;
they are not held-out performance evidence and never supply tuning features.


In [ ]:
import json, os
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

REPO = Path.cwd()
if not (REPO / "run_revision.py").is_file():
    REPO = REPO.parent
assert (
    REPO / "run_revision.py"
).is_file(), "Start Jupyter in the repository or manuscript directory"
import sys

if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from revision.data import TASK_PROTOCOL, TASK_TITLES, TASK3_CAPTION


def csv(path, limit=None):
    path = Path(path)
    if not path.is_file():
        display(Markdown(f"Pending/unavailable: `{path.name}`"))
        return pd.DataFrame()
    try:
        table = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        table = pd.DataFrame()
    display(table if limit is None else table.head(limit))
    return table


def images(paths):
    for path in paths:
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))
        if "cancer_vs_lr_hr" in str(path):
            display(Markdown(TASK3_CAPTION.replace("\n", " ")))


QUALITY = REPO / os.environ.get("IIVV_QUALITY", "results/revision-quality-v3")
quality_run = json.loads((QUALITY / "run_manifest.json").read_text())
assert (
    quality_run["config"].get("task_protocol") == TASK_PROTOCOL
), "Obsolete task protocol"
assert quality_run["status"] in {
    "quality_completed",
    "completed",
}, "Quality computation is incomplete"
assert not quality_run.get("synthetic"), "Synthetic quality is not clinical evidence"
from revision.data_quality import verify_quality

quality_manifest = verify_quality(QUALITY, quality_run["config"])
TASKS = quality_run["config"]["tasks"]
display(
    {
        "status": quality_run["status"],
        "cohort": quality_run["input_summary"],
        "protocol": TASK_PROTOCOL,
    }
)

## Cohort, source agreement and clinical metadata

In [ ]:
display(json.loads((QUALITY / "analysis_status.json").read_text()))
for name in [
    "site_inclusion_audit.csv",
    "source_quality_audit.csv",
    "class_by_location.csv",
    "covariate_cluster_associations.csv",
    "quality_cluster_associations.csv",
]:
    csv(QUALITY / "quality" / name)
display(json.loads((QUALITY / "quality/clinical_metadata_status.json").read_text()))
csv(QUALITY / "exploration/patient_pathology_overlap.csv")
csv(QUALITY / "exploration/cohort_pathology_counts.csv")

## Missingness, spectrum properties and acquisition rejection

Acquisition rejection percentages differ from missing coordinates in averaged spectra.
Patients with multiple diagnoses do not demonstrate mixed histology within one footprint.


In [ ]:
for name in [
    "site_quality.csv",
    "frequency_quality.csv",
    "pattern_quality.csv",
    "mask_frequency_quality.csv",
]:
    csv(QUALITY / "exploration" / name, 40)
images([QUALITY / "exploration/quality-overview.png"])
images([QUALITY / "exploration/mask-frequency-quality.png"])

## Task populations and quality retention

Cutoffs here are descriptive audits; they do not select classifier preprocessing.
Task 3 Low-grade dysplasia includes normal/benign tissue and mild dysplasia;
High-grade dysplasia includes moderate/severe dysplasia and CIS. Healthy sites are excluded.


In [ ]:
retention = csv(QUALITY / "exploration/task_quality_retention.csv")
for task in TASKS:
    display(Markdown(f"### {TASK_TITLES[task]}"))
    csv(QUALITY / "exploration" / f"{task}-inclusion.csv")
    csv(QUALITY / "exploration" / f"{task}-label-audit.csv", 30)
    images([QUALITY / "exploration" / f"{task}-quality-retention.png"])

## Unsupervised 2D PCA and exploratory MI-guided 2D PCA

One row represents a measurement site. Each frequency is summarized over finite IIVV
magnitudes, then log transformed. Patient-balanced mean imputation, scaling and SVD
are descriptive operations on this cohort. MI uses labels; the MI-guided plot must
not be interpreted as independent evidence of separation.


In [ ]:
for task in TASKS:
    display(Markdown(f"### {TASK_TITLES[task]}"))
    images(sorted((QUALITY / "exploration").glob(f"{task}-*pca.png")))
    csv(QUALITY / "exploration" / f"{task}-pca-loadings.csv")
    csv(QUALITY / "exploration" / f"{task}-mi-pca-loadings.csv")
    display(
        json.loads((QUALITY / "exploration" / f"{task}-mi-analysis.json").read_text())
    )
    ranking = csv(QUALITY / "exploration" / f"{task}-mutual-information.csv")
    images([QUALITY / "exploration" / f"{task}-mutual-information.png"])
    csv(QUALITY / "exploration" / f"{task}-mi-patient-draws.csv", 30)

## Repeatability of saved test means

ICC(2,1) and whole-patient bootstrap intervals describe the supplied test means.
They do not establish individual-burst repeatability or controlled contact conditions.


In [ ]:
display(json.loads((QUALITY / "repeatability/status.json").read_text()))
csv(QUALITY / "repeatability/icc_and_inter_site_variability.csv")
csv(QUALITY / "repeatability/site_repeatability.csv", 40)
if (QUALITY / "exploration/saved-test-mean-icc.png").is_file():
    images([QUALITY / "exploration/saved-test-mean-icc.png"])

## Planned outer/inner CV demographics and raw pathology

In [ ]:
for task in TASKS:
    display(Markdown(f"### {TASK_TITLES[task]}"))
    for name in [
        "cv_population_summary.csv",
        "cv_pathology_breakdown.csv",
        "cv_demographics_availability.csv",
        "cv_demographics_categorical.csv",
        "cv_demographics_numeric.csv",
        "cv_patient_groups.csv",
    ]:
        csv(QUALITY / "planned-splits" / task / name, 40)